# Notebook 03 — Molecular Descriptors: Quantifying Chemical Properties for Computation

In the wet lab, you characterize molecules by measuring properties — LogP via shake-flask, melting point on a hot stage, pKa by titration. Each measurement gives you a **number** that captures one facet of molecular behavior.

**Molecular descriptors** are the computational analog. They translate a molecular graph (atoms + bonds) into numerical features — the bridge between chemistry and machine learning. Instead of running an experiment, you calculate a value from the structure itself.

This notebook covers:
1. **Physicochemical descriptors** — LogP, TPSA, HBD/HBA (the ones you already know from the bench)
2. **Topological descriptors** — rotatable bonds, Fsp3, ring counts (encoding shape and flexibility)
3. **Electronic descriptors** — Gasteiger partial charges (inductive effects, quantified)
4. **Bulk computation** — building descriptor DataFrames for QSAR/ML pipelines
5. **QED** — a composite drug-likeness score that wraps multiple descriptors into one number

By the end, you will be able to go from a SMILES string to a feature vector suitable for any downstream modeling task.

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
import pandas as pd

## Setting Up a Drug Panel

We will work with a diverse set of well-known drugs throughout this notebook. These span a range of molecular weight, polarity, and therapeutic class — so you can see how descriptors differentiate structurally distinct molecules.

| Drug | Class | Why it's interesting |
|------|-------|---------------------|
| Aspirin | NSAID | Small, acidic, highly polar |
| Caffeine | Stimulant | Purine scaffold, multiple N-methyls |
| Ibuprofen | NSAID | Lipophilic, flexible chain |
| Metformin | Antidiabetic | Very small, highly polar biguanide |
| Atorvastatin | Statin | Large, multi-ring, many H-bond sites |
| Diazepam | Benzodiazepine | Chlorinated, fused heterocycle |
| Omeprazole | PPI | Sulfoxide, multiple heteroatoms |
| Acetaminophen | Analgesic | Simple phenol + amide |

In [2]:
drug_data = {
    "Aspirin":       "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine":      "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Ibuprofen":     "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Metformin":     "CN(C)C(=N)NC(=N)N",
    "Atorvastatin":  "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
    "Diazepam":      "CN1C(=O)CN=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C21",
    "Omeprazole":    "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
}

mols = {}
for name, smi in drug_data.items():
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mols[name] = mol

print(f"Successfully parsed {len(mols)}/{len(drug_data)} molecules")

Successfully parsed 8/8 molecules


## 1. Physicochemical Descriptors

These are the descriptors you already have intuition for from the bench. Each one has a direct experimental counterpart:

### LogP (Partition Coefficient)
Remember the **shake-flask experiment**? You dissolve a compound in a separating funnel with octanol and water, shake, let the layers separate, and measure the concentration ratio. `LogP = log10([octanol] / [water])`.

- **Positive LogP** → lipophilic (prefers the organic phase) → good membrane permeability
- **Negative LogP** → hydrophilic (prefers water) → better solubility, worse permeability
- RDKit uses the **Wildman-Crippen** method: atom-based contributions summed over the molecule. Fast and surprisingly accurate for drug-like space.

### TPSA (Topological Polar Surface Area)
The sum of surface contributions from polar atoms (N, O, and their attached H atoms). This is a 2D approximation of what you would compute from a 3D molecular surface.

- **< 140 A^2** → generally good oral absorption
- **< 90 A^2** → can cross the blood-brain barrier
- Think of it as a quick proxy for "how much of the molecule's surface is available for hydrogen bonding with water."

### MR (Molar Refractivity)
Related to molecular volume and polarizability. In the lab, you would measure this with a refractometer. Computationally, it uses atom-based increments (same Wildman-Crippen scheme as LogP). Useful as a steric descriptor in QSAR.

### HBD / HBA (Hydrogen Bond Donors / Acceptors)
Exactly what you count when evaluating Lipinski's Rule of Five:
- **HBD**: OH and NH groups (atoms that can donate a hydrogen bond)
- **HBA**: N and O atoms (atoms with lone pairs that can accept a hydrogen bond)

In [4]:
for name, mol in mols.items():
    print(f"\n{name}")
    print(f"  LogP:  {Descriptors.MolLogP(mol):.2f}")
    print(f"  TPSA:  {Descriptors.TPSA(mol):.1f} A^2")
    print(f"  MR:    {Descriptors.MolMR(mol):.1f}")
    print(f"  HBD:   {rdMolDescriptors.CalcNumHBD(mol)}")
    print(f"  HBA:   {rdMolDescriptors.CalcNumHBA(mol)}")


Aspirin
  LogP:  1.31
  TPSA:  63.6 A^2
  MR:    44.7
  HBD:   1
  HBA:   3

Caffeine
  LogP:  -1.03
  TPSA:  61.8 A^2
  MR:    51.2
  HBD:   0
  HBA:   3

Ibuprofen
  LogP:  3.07
  TPSA:  37.3 A^2
  MR:    61.0
  HBD:   1
  HBA:   1

Metformin
  LogP:  -1.03
  TPSA:  89.0 A^2
  MR:    36.5
  HBD:   4
  HBA:   2

Atorvastatin
  LogP:  6.31
  TPSA:  111.8 A^2
  MR:    157.2
  HBD:   4
  HBA:   4

Diazepam
  LogP:  3.15
  TPSA:  32.7 A^2
  MR:    81.8
  HBD:   0
  HBA:   2

Omeprazole
  LogP:  2.90
  TPSA:  77.1 A^2
  MR:    93.0
  HBD:   1
  HBA:   5

Acetaminophen
  LogP:  1.35
  TPSA:  49.3 A^2
  MR:    42.4
  HBD:   2
  HBA:   2


**What to notice:**
- **Metformin** has strongly negative LogP — it is one of the most hydrophilic oral drugs. It relies on organic cation transporters (OCTs) rather than passive diffusion to cross membranes.
- **Ibuprofen** has the highest LogP — consistent with its lipophilic isobutyl + phenyl scaffold.
- **Atorvastatin** has the highest TPSA and most HBD/HBA — yet it is still orally bioavailable, partly because of active transport via OATP1B1.

## 2. Topological Descriptors

Topological descriptors capture the **shape and connectivity** of a molecule without requiring 3D coordinates. They work on the molecular graph alone — atoms are nodes, bonds are edges.

### Rotatable Bonds
A bond is "rotatable" if it is a single bond, not in a ring, and not connected to a terminal heavy atom. More rotatable bonds means:
- **Greater conformational flexibility** — the molecule can adopt many shapes
- **Higher entropy penalty on binding** — a flexible molecule loses more entropy when it locks into a binding pose
- Rule of thumb: **< 10 rotatable bonds** for good oral bioavailability (Veber's rules)

### Ring Count
Ring systems provide **rigidity** and **planarity**. Aromatic rings stack via pi-pi interactions. Fused ring systems (like benzodiazepines or purines) constrain the molecule into well-defined shapes, reducing the conformational search space.

### Fsp3 — Fraction of sp3 Carbons
This is the "escape from flatland" descriptor (Lovering et al., 2009). It measures what fraction of carbon atoms are sp3-hybridized (tetrahedral) versus sp2 (planar).

- **Higher Fsp3** → more three-dimensionality → better clinical success rates
- **Lower Fsp3** → flatter molecule → more likely to have selectivity and toxicity issues
- Many early combinatorial chemistry libraries were "too flat" — the field has since moved toward sp3-rich scaffolds.

### Aromatic Rings
Counted separately from total rings. Too many aromatic rings (> 3) is associated with poor developability — increased promiscuity, hERG liability, and poor solubility.

In [5]:
for name, mol in mols.items():
    print(f"\n{name}")
    print(f"  Rotatable bonds: {rdMolDescriptors.CalcNumRotatableBonds(mol)}")
    print(f"  Ring count:      {rdMolDescriptors.CalcNumRings(mol)}")
    print(f"  Aromatic rings:  {rdMolDescriptors.CalcNumAromaticRings(mol)}")
    print(f"  Fsp3:            {rdMolDescriptors.CalcFractionCSP3(mol):.2f}")


Aspirin
  Rotatable bonds: 2
  Ring count:      1
  Aromatic rings:  1
  Fsp3:            0.11

Caffeine
  Rotatable bonds: 0
  Ring count:      2
  Aromatic rings:  2
  Fsp3:            0.38

Ibuprofen
  Rotatable bonds: 4
  Ring count:      1
  Aromatic rings:  1
  Fsp3:            0.46

Metformin
  Rotatable bonds: 0
  Ring count:      0
  Aromatic rings:  0
  Fsp3:            0.50

Atorvastatin
  Rotatable bonds: 12
  Ring count:      4
  Aromatic rings:  4
  Fsp3:            0.27

Diazepam
  Rotatable bonds: 1
  Ring count:      3
  Aromatic rings:  2
  Fsp3:            0.12

Omeprazole
  Rotatable bonds: 5
  Ring count:      3
  Aromatic rings:  3
  Fsp3:            0.29

Acetaminophen
  Rotatable bonds: 1
  Ring count:      1
  Aromatic rings:  1
  Fsp3:            0.12


**What to notice:**
- **Metformin** has 0 rings and 0 aromatic rings — it is a purely linear, open-chain molecule. Its Fsp3 is also 0.33, reflecting the sp2 guanidinium groups.
- **Atorvastatin** has the most rotatable bonds (12) — consistent with its flexible dihydroxyheptanoic acid side chain.
- **Caffeine** and **Diazepam** have Fsp3 near zero — they are essentially flat, fused heterocyclic systems.
- **Ibuprofen** has the highest Fsp3 (0.62) — its isobutyl group and chiral center give it three-dimensionality.

## 3. Electronic Descriptors — Gasteiger Partial Charges

### Chemistry refresher: Partial charges and inductive effects

In organic chemistry, you learn that atoms in a molecule carry **partial charges** due to electronegativity differences. A carbonyl oxygen pulls electron density away from carbon; a nitro group withdraws electron density through the pi system.

**Gasteiger charges** (also called Gasteiger-Marsili charges) approximate these partial charges using an iterative **electronegativity equalization** algorithm:
1. Start with atomic electronegativities.
2. On each iteration, redistribute charge along each bond proportional to the electronegativity difference.
3. Dampen the transfer on each step (convergence factor).
4. After ~6 iterations, the charges stabilize.

This captures **inductive effects** (through-bond polarization) — the same concept you use when predicting acidity trends or drawing resonance structures. It does *not* capture resonance effects well, but it is fast and works directly on the molecular graph without 3D coordinates.

Let us compute Gasteiger charges for aspirin and see how the electron density distributes across its atoms.

In [6]:
# Compute Gasteiger charges for Aspirin
aspirin = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")
AllChem.ComputeGasteigerCharges(aspirin)

print("Gasteiger partial charges for Aspirin")
print("=" * 45)
for atom in aspirin.GetAtoms():
    charge = float(atom.GetProp("_GasteigerCharge"))
    symbol = atom.GetSymbol()
    idx = atom.GetIdx()
    # Flag atoms with significant charge
    flag = ""
    if charge > 0.2:
        flag = " <-- electrophilic"
    elif charge < -0.3:
        flag = " <-- nucleophilic"
    print(f"  Atom {idx:2d} ({symbol:2s}): {charge:+.3f}{flag}")

Gasteiger partial charges for Aspirin
  Atom  0 (C ): +0.034
  Atom  1 (C ): +0.308 <-- electrophilic
  Atom  2 (O ): -0.252
  Atom  3 (O ): -0.426 <-- nucleophilic
  Atom  4 (C ): +0.145
  Atom  5 (C ): -0.018
  Atom  6 (C ): -0.059
  Atom  7 (C ): -0.061
  Atom  8 (C ): -0.044
  Atom  9 (C ): +0.102
  Atom 10 (C ): +0.339 <-- electrophilic
  Atom 11 (O ): -0.246
  Atom 12 (O ): -0.478 <-- nucleophilic


**Interpreting the output:**
- The **carbonyl carbons** (C=O) carry the largest positive charges — they are electrophilic centers, exactly as you would predict from drawing resonance structures.
- The **oxygens** carry negative charges — they are the nucleophilic/basic sites.
- The **aromatic carbons** have small charges near zero — the pi system distributes charge evenly.

These per-atom charges can be used as features in graph neural networks or summarized into molecular-level descriptors (e.g., max positive charge, min negative charge, charge range).

## 4. Bulk Computation with Pandas

For QSAR modeling and machine learning, you need descriptor values as columns in a DataFrame — one row per molecule, one column per descriptor. RDKit makes this straightforward.

### Curated descriptor set
First, let us compute a hand-picked set of the most commonly used descriptors. This is what you would typically start with for a drug discovery project.

In [7]:
# Build a descriptor DataFrame with a curated set of descriptors
rows = []
for name, mol in mols.items():
    row = {"Name": name}
    row["MW"] = Descriptors.MolWt(mol)
    row["LogP"] = Descriptors.MolLogP(mol)
    row["TPSA"] = Descriptors.TPSA(mol)
    row["HBD"] = rdMolDescriptors.CalcNumHBD(mol)
    row["HBA"] = rdMolDescriptors.CalcNumHBA(mol)
    row["RotBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    row["Rings"] = rdMolDescriptors.CalcNumRings(mol)
    row["Fsp3"] = rdMolDescriptors.CalcFractionCSP3(mol)
    rows.append(row)

df = pd.DataFrame(rows).set_index("Name")
df

,MW,LogP,TPSA,HBD,HBA,RotBonds,Rings,Fsp3
Name,,,,,,,,
Aspirin,180.159,1.31010,63.60,1,3,2,1,0.111111
Caffeine,194.194,-1.02930,61.82,0,3,0,2,0.375000
Ibuprofen,206.285,3.07320,37.30,1,1,4,1,0.461538
Metformin,129.167,-1.03416,88.99,4,2,0,0,0.500000
Atorvastatin,558.650,6.31360,111.79,4,4,12,4,0.272727
Diazepam,284.746,3.15380,32.67,0,2,1,3,0.125000
Omeprazole,345.424,2.89974,77.10,1,5,5,3,0.294118
Acetaminophen,151.165,1.35060,49.33,2,2,1,1,0.125000


### Computing ALL descriptors at once

RDKit ships with `Descriptors.descList` — a list of `(name, function)` tuples for every available descriptor. This is the "kitchen sink" approach: compute everything and let feature selection sort it out.

In [8]:
# How many descriptors does RDKit provide?
desc_names = [name for name, _ in Descriptors.descList]
print(f"Total available descriptors: {len(desc_names)}")
print(f"\nFirst 20 descriptor names:\n{desc_names[:20]}")
print(f"\nLast 10 descriptor names:\n{desc_names[-10:]}")

Total available descriptors: 217

First 20 descriptor names:
['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW']

Last 10 descriptor names:
['fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene', 'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene', 'fr_unbrch_alkane', 'fr_urea']


In [9]:
# Compute ALL descriptors for one molecule as a demonstration
calc = {name: func(mols["Aspirin"]) for name, func in Descriptors.descList}
print(f"Computed {len(calc)} descriptors for Aspirin")
print(f"\nFirst 10 descriptors:")
for name, value in list(calc.items())[:10]:
    print(f"  {name:30s} = {value}")

Computed 217 descriptors for Aspirin

First 10 descriptors:
  MaxAbsEStateIndex              = 10.611948223733938
  MaxEStateIndex                 = 10.611948223733938
  MinAbsEStateIndex              = 0.01601851851851821
  MinEStateIndex                 = -1.1140277777777776
  qed                            = 0.5501217966938848
  SPS                            = 9.307692307692308
  MolWt                          = 180.15899999999996
  HeavyAtomMolWt                 = 172.09499999999997
  ExactMolWt                     = 180.042258736
  NumValenceElectrons            = 68


In [10]:
# Build the full descriptor DataFrame for all molecules
all_rows = []
for name, mol in mols.items():
    row = {"Name": name}
    for desc_name, func in Descriptors.descList:
        try:
            row[desc_name] = ee(mol)
        except Exception:
            row[desc_name] = None
    all_rows.append(row)

df_all = pd.DataFrame(all_rows).set_index("Name")
print(f"DataFrame shape: {df_all.shape}  (molecules x descriptors)")
print(f"\nDescriptor statistics:")
df_all.describe().T.head(10)

DataFrame shape: (8, 217)  (molecules x descriptors)

Descriptor statistics:


,count,mean,std,min,25%,50%,75%,max
MaxAbsEStateIndex,8.0,11.164398,2.036969,7.067778,10.590078,11.217231,12.169287,14.048315
MaxEStateIndex,8.0,11.164398,2.036969,7.067778,10.590078,11.217231,12.169287,14.048315
MinAbsEStateIndex,8.0,0.174264,0.148255,0.016019,0.077480,0.114264,0.298469,0.418426
MinEStateIndex,8.0,-0.636654,0.519884,-1.309847,-1.130569,-0.565829,-0.189401,-0.028237
qed,8.0,0.559705,0.245624,0.162760,0.466043,0.572574,0.774841,0.821600
SPS,8.0,11.281025,2.211516,8.222222,9.276224,11.702381,12.748780,14.600000
MolWt,8.0,256.223750,141.188837,129.167000,172.910500,200.239500,299.915500,558.650000
HeavyAtomMolWt,8.0,240.725750,132.886965,118.079000,164.594500,186.127500,285.299500,523.370000
ExactMolWt,8.0,255.982180,141.063227,129.101445,172.797526,200.105528,299.332409,558.253000
NumValenceElectrons,8.0,96.750000,53.022233,52.000000,65.500000,78.000000,106.500000,214.000000


**Practical note:** When computing all descriptors for large datasets (thousands of molecules), some descriptors may fail or return `NaN` for certain chemistries. Always wrap descriptor computation in a try/except and check for missing values before feeding into ML models.

Also be aware of **highly correlated descriptors** — for example, `MolWt` and `HeavyAtomMolWt` will be nearly perfectly correlated. Feature selection or PCA is essential before modeling.

## 5. QED — Quantitative Estimate of Drug-likeness

### Chemistry refresher: From Lipinski to QED

You know **Lipinski's Rule of Five** — a set of hard cutoffs (MW < 500, LogP < 5, HBD < 5, HBA < 10) that predict oral bioavailability. The problem? Hard cutoffs throw away nuance. A molecule with MW=501 is not meaningfully different from one with MW=499, yet Lipinski flags only the first.

**QED** (Bickerton et al., 2012) improves on this by combining eight molecular properties into a single continuous score between 0 and 1:

| Property | What it measures |
|----------|-----------------|
| MW | Molecular weight |
| ALOGP | Lipophilicity |
| HBA | H-bond acceptors |
| HBD | H-bond donors |
| PSA | Polar surface area |
| ROTB | Rotatable bonds |
| AROM | Aromatic rings |
| ALERTS | Structural alerts (unwanted functional groups) |

Each property is passed through a **desirability function** (an asymmetric double sigmoidal) fitted to the distribution of that property across known drugs. The QED score is the geometric mean of all eight desirability values.

- **QED near 1.0** → the molecule resembles known oral drugs across all eight dimensions
- **QED near 0.0** → the molecule deviates significantly from drug-like space
- QED is **not** a hard filter — it is a continuous, multi-parameter optimization score

In [11]:
# Compute QED scores and the underlying property breakdown
print(f"{'Drug':15s}  {'QED':>5s}  {'MW':>6s}  {'LogP':>6s}  {'HBA':>4s}  {'HBD':>4s}  {'PSA':>6s}  {'RotB':>4s}  {'Arom':>4s}")
print("-" * 75)

for name, mol in mols.items():
    qed_score = QED.qed(mol)
    props = QED.properties(mol)
    print(
        f"{name:15s}  {qed_score:.3f}  {props.MW:6.1f}  {props.ALOGP:6.2f}  "
        f"{props.HBA:4d}  {props.HBD:4d}  {props.PSA:6.1f}  "
        f"{props.ROTB:4d}  {props.AROM:4d}"
    )

Drug               QED      MW    LogP   HBA   HBD     PSA  RotB  Arom
---------------------------------------------------------------------------
Aspirin          0.550   180.2    1.31     4     1    63.6     2     1
Caffeine         0.538   194.2   -1.03     3     0    61.8     0     2
Ibuprofen        0.822   206.3    3.07     2     1    37.3     4     1
Metformin        0.249   129.2   -1.03     3     4    89.0     0     0
Atorvastatin     0.163   558.7    6.31     5     4   111.8    12     4
Diazepam         0.792   284.7    3.15     1     0    32.7     1     2
Omeprazole       0.769   345.4    2.90     5     1    77.1     5     3
Acetaminophen    0.595   151.2    1.35     2     2    49.3     1     1


**What to notice:**
- **Acetaminophen** and **Diazepam** likely score highest — they are small, simple, well-balanced molecules that sit squarely in drug-like space.
- **Atorvastatin** scores lower — it is large (MW > 500), has many rotatable bonds, and many H-bond donors/acceptors. It is a successful drug, but it falls outside the "typical" oral drug profile.
- **Metformin** may also score lower despite being a blockbuster drug — it is very small and very polar, which is unusual in the drug-like property distributions QED was trained on.

This illustrates a key point: **QED measures similarity to the average oral drug, not actual efficacy.** Many successful drugs are outliers.

## Summary

### What we covered

| Descriptor type | Examples | Speed | Use case |
|----------------|----------|-------|----------|
| **Physicochemical** | LogP, TPSA, MR, HBD/HBA | Very fast | Lipinski filtering, ADME prediction |
| **Topological** | Rotatable bonds, Fsp3, ring count | Very fast | Complexity analysis, lead-likeness |
| **Electronic** | Gasteiger charges | Fast | Reactivity prediction, per-atom features |
| **Composite** | QED | Fast | Multi-parameter optimization, triage |
| **Full descriptor set** | 200+ from `Descriptors.descList` | Moderate | QSAR modeling, ML feature matrices |

### Key takeaways

1. **Descriptors are the bridge between chemistry and computation.** Every ML model needs numerical input — descriptors convert molecular structure into feature vectors.

2. **Start with interpretable descriptors.** LogP, TPSA, MW, and HBD/HBA have direct physical meaning. Use these first before reaching for the full 200+ descriptor set.

3. **Watch for correlations.** Many descriptors are highly redundant. Always perform feature selection or dimensionality reduction before modeling.

4. **No single descriptor tells the whole story.** QED is useful precisely because it combines multiple properties — but even QED misses successful drugs like metformin that are outliers in property space.

### What's next

In **Notebook 04**, we will move from individual descriptors to **molecular fingerprints** — fixed-length binary or count vectors that encode substructural features. Fingerprints are the other major molecular representation used in cheminformatics, and they complement descriptor-based approaches for similarity searching, clustering, and ML.